<a href="https://colab.research.google.com/github/Farrukh776/agentic-ai-hands-on/blob/main/rag/rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Build a RAG System — from Retrieval to Agentic RAG

Work through the tasks **in order**. Each task builds on the previous one. Fill in every cell marked `# TODO`,
run it, and answer the questions in the lab manual.

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the setup cells.

## Task 0 — Setup

In [1]:
!pip -q install sentence-transformers rank_bm25

In [2]:
import re, textwrap, numpy as np, torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi

# Embedding model (small and fast)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed(texts):
    """Return one normalised embedding vector per text (so dot product = cosine similarity)."""
    return embedder.encode(texts, normalize_embeddings=True)

# Small instruction-tuned LLM that fits on a free T4 GPU
LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype=torch.float16, device_map="auto")

def generate(messages, max_new_tokens=200):
    """messages = [{"role": "system"/"user"/"assistant", "content": "..."}] -> reply text"""
    text = tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(generate([{"role": "user", "content": "Say hello in five words."}]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (14 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Hello there! How can I assist you today?


In [3]:
HANDBOOK = "SAMPLE UNIVERSITY STUDENT HANDBOOK (for lab use only)\n\n1. Attendance\nStudents must attend at least 80% of the lectures of every course. A student whose attendance in a course falls below 80% is not allowed to sit the final examination of that course and is awarded an SA (short attendance) grade. Attendance is recorded by the instructor in every lecture. Medical leave must be supported by a certificate from the university medical officer and submitted to the academic office within seven days of returning to class. Approved medical leave is not counted as absence, but a student cannot be excused for more than three weeks of lectures in a semester.\n\n2. Examinations\nEach course has two sessional examinations and one final examination. Sessional examinations are held in the sixth and eleventh weeks of the semester. A student who misses a sessional examination for a genuine reason may apply for a retake through the head of department within three working days. There is no retake of a missed final examination; instead, the student may apply for an incomplete (I) grade, which must be cleared in the next regular semester. Mobile phones, smart watches and any notes are not permitted in the examination hall.\n\n3. Grading\nGrades are awarded on a relative scale within each course section. The minimum passing grade is D. A student must maintain a cumulative grade point average (CGPA) of at least 2.0 to remain in good standing. A student whose CGPA falls below 2.0 is placed on academic probation for the next semester and may register for a maximum of four courses. Two consecutive semesters on probation lead to dismissal from the program.\n\n4. Fee Refund Policy\nTuition fees are refunded in full upon withdrawal before the commencement of classes. A student who withdraws during the first week of classes receives a 75% refund of tuition fees, and a student who withdraws during the second week receives a 50% refund. No tuition fee is refunded after the second week of classes. Admission fees and security deposits are not refundable, except that the security deposit is returned when the student graduates or leaves the university with all dues cleared. To request a refund, the student must submit form F-4015 to the accounts office together with the original fee receipt. Refunds are processed within thirty days.\n\n5. Library\nStudents may borrow up to three books at a time for a loan period of fourteen days. A book may be renewed once if no other student has reserved it. A fine of fifty rupees per day is charged for each overdue book. Reference books, theses and journals cannot be taken out of the library. The library is open from 8 am to 8 pm on weekdays and from 9 am to 5 pm on Saturdays; it remains closed on Sundays and public holidays. Lost books must be replaced or paid for at twice the current price.\n\n6. Academic Integrity\nPlagiarism, cheating and the use of unauthorised aids, including AI tools where the instructor has not allowed them, are serious offences. A first offence of plagiarism in an assignment results in zero marks for that assignment and a written warning. A second offence, or any case of cheating in an examination, is referred to the disciplinary committee, which may award an F grade in the course or suspend the student for one semester. All submitted work may be checked with similarity-detection software.\n\n7. Transcripts and Degrees\nAn official transcript can be requested from the examinations office by submitting the transcript request form and paying the prescribed fee. Transcripts are issued within five working days; an urgent transcript can be issued within one working day for an additional fee. Degrees are awarded at the annual convocation. A student who cannot attend the convocation may collect the degree from the examinations office afterwards by presenting the original national identity card.\n\n8. Hostel\nHostel accommodation is available for students whose home is outside the city, subject to availability. Applications are invited at the start of each academic year and rooms are allotted on merit. Hostel residents must return by 10 pm unless they have written permission from the warden. Cooking appliances are not permitted in the rooms. Visitors may meet residents only in the common room between 4 pm and 7 pm.\n"
print(len(HANDBOOK.split()), 'words')
print(HANDBOOK[:600])

718 words
SAMPLE UNIVERSITY STUDENT HANDBOOK (for lab use only)

1. Attendance
Students must attend at least 80% of the lectures of every course. A student whose attendance in a course falls below 80% is not allowed to sit the final examination of that course and is awarded an SA (short attendance) grade. Attendance is recorded by the instructor in every lecture. Medical leave must be supported by a certificate from the university medical officer and submitted to the academic office within seven days of returning to class. Approved medical leave is not counted as absence, but a student cannot be excused


## Task 1 — Chunking
Split the handbook into chunks of `size` words, where neighbouring chunks overlap by `overlap` words.

In [4]:
def chunk_words(text, size=100, overlap=20):
    words = text.split()
    step = size - overlap
    chunks = []
    for start in range(0, len(words), step):
        piece = words[start:start + size]
        chunks.append(" ".join(piece))
        if start + size >= len(words):      # this chunk reached the end of the text
            break
    return chunks

chunks = chunk_words(HANDBOOK, size=100, overlap=20)
print(len(chunks), "chunks")
for i, c in enumerate(chunks[:3]):
    print(f"--- chunk {i} ---\n{textwrap.fill(c, 100)}\n")

9 chunks
--- chunk 0 ---
SAMPLE UNIVERSITY STUDENT HANDBOOK (for lab use only) 1. Attendance Students must attend at least
80% of the lectures of every course. A student whose attendance in a course falls below 80% is not
allowed to sit the final examination of that course and is awarded an SA (short attendance) grade.
Attendance is recorded by the instructor in every lecture. Medical leave must be supported by a
certificate from the university medical officer and submitted to the academic office within seven
days of returning to class. Approved medical leave is not counted as absence, but a student cannot
be excused

--- chunk 1 ---
seven days of returning to class. Approved medical leave is not counted as absence, but a student
cannot be excused for more than three weeks of lectures in a semester. 2. Examinations Each course
has two sessional examinations and one final examination. Sessional examinations are held in the
sixth and eleventh weeks of the semester. A student who misses a

In [6]:
#1B=chunks that do not start with a capital letter or end without a full stop
for i, c in enumerate(chunks):
    print(i, "| starts:", c[:40], "| ends:", c[-40:])

0 | starts: SAMPLE UNIVERSITY STUDENT HANDBOOK (for  | ends: absence, but a student cannot be excused
1 | starts: seven days of returning to class. Approv | ends: apply for an incomplete (I) grade, which
2 | starts: There is no retake of a missed final exa | ends: r the next semester and may register for
3 | starts: A student whose CGPA falls below 2.0 is  | ends: Admission fees and security deposits are
4 | starts: a 50% refund. No tuition fee is refunded | ends: e if no other student has reserved it. A
5 | starts: a loan period of fourteen days. A book m | ends: of unauthorised aids, including AI tools
6 | starts: for at twice the current price. 6. Acade | ends: nd Degrees An official transcript can be
7 | starts: one semester. All submitted work may be  | ends: e for students whose home is outside the
8 | starts: by presenting the original national iden | ends: n the common room between 4 pm and 7 pm.


## Task 2 — Embed and Retrieve
Embed every chunk once, then retrieve the top-k chunks for a question by cosine similarity.

In [7]:
chunk_emb = embed(chunks)          # shape: (number of chunks, 384)

def retrieve(question, k=3, chunks=chunks, chunk_emb=chunk_emb):
    q = embed([question])[0]
    scores = chunk_emb @ q
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]

for i, s in retrieve("How many books can I borrow from the library?"):
    print(f"[{i}] score={s:.3f}  {chunks[i][:110]}...")

[5] score=0.530  a loan period of fourteen days. A book may be renewed once if no other student has reserved it. A fine of fift...
[4] score=0.438  a 50% refund. No tuition fee is refunded after the second week of classes. Admission fees and security deposit...
[3] score=0.157  A student whose CGPA falls below 2.0 is placed on academic probation for the next semester and may register fo...


In [8]:
#2B
for i, s in retrieve("Can I bring my laptop charger into the exam?"):
    print(f"[{i}] score={s:.3f}  {chunks[i][:110]}...")

[7] score=0.366  one semester. All submitted work may be checked with similarity-detection software. 7. Transcripts and Degrees...
[0] score=0.364  SAMPLE UNIVERSITY STUDENT HANDBOOK (for lab use only) 1. Attendance Students must attend at least 80% of the l...
[2] score=0.321  There is no retake of a missed final examination; instead, the student may apply for an incomplete (I) grade, ...


## Task 3 — Evaluate Retrieval (Recall@k)
Each test question comes with an **evidence** string that appears only in the passage containing the answer.
A question counts as a hit if any of the top-k retrieved chunks contains its evidence.

In [9]:
EVAL = [
    ("What attendance do I need to sit the final exam?",          "at least 80%"),
    ("Which form do I submit to get a fee refund?",                "F-4015"),
    ("How long can I keep a library book?",                         "loan period of fourteen days"),
    ("What happens the first time a student plagiarises?",          "written warning"),
    ("How quickly can I get an urgent transcript?",                 "within one working day"),
    ("What happens if my CGPA falls below 2.0?",                    "academic probation"),
]

def recall_at_k(retrieve_fn, chunks, k=3):
    hits = 0
    for question, evidence in EVAL:
        retrieved = [chunks[i] for i, _ in retrieve_fn(question, k)]
        if any(evidence in c for c in retrieved):
            hits += 1
    return hits / len(EVAL)

print("Recall@3 =", recall_at_k(retrieve, chunks, k=3))

Recall@3 = 1.0


In [10]:
# Compare chunk sizes
for size in [40, 100, 250]:
    ch = chunk_words(HANDBOOK, size=size, overlap=size // 5)
    em = embed(ch)
    r = lambda q, k, ch=ch, em=em: retrieve(q, k, chunks=ch, chunk_emb=em)
    print(f"size={size:4d}  chunks={len(ch):3d}  Recall@1={recall_at_k(r, ch, 1):.2f}  Recall@3={recall_at_k(r, ch, 3):.2f}")

size=  40  chunks= 23  Recall@1=1.00  Recall@3=1.00
size= 100  chunks=  9  Recall@1=1.00  Recall@3=1.00
size= 250  chunks=  4  Recall@1=1.00  Recall@3=1.00


## Task 4 — Generate a Grounded Answer with Citations

In [11]:
SYSTEM = ("Answer the question using ONLY the numbered context passages. Cite passages as [1], [2]. "
          "If the answer is not in the context, say exactly: I could not find this in the documents.")

def build_prompt(question, passages):
    numbered = "\n".join(f"[{i}] {p}" for i, p in enumerate(passages, start=1))
    return f"Context:\n{numbered}\n\nQuestion: {question}"

def rag_answer(question, k=3, retrieve_fn=retrieve):
    passages = [chunks[i] for i, _ in retrieve_fn(question, k)]
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": build_prompt(question, passages)}]
    return generate(messages), passages

answer, used = rag_answer("Can I take a reference book home?")
print(answer)

No. [1]


In [12]:
# A question whose answer is NOT in the handbook
print(rag_answer("What is the hostel fee for international students?")[0])

I could not find this in the documents.


In [13]:
#4c
SYSTEM_ORIG = SYSTEM
SYSTEM = "Answer the question using ONLY the numbered context passages. Cite passages as [1], [2]."
print("WITHOUT refusal sentence:", rag_answer("What is the hostel fee for international students?")[0])
SYSTEM = SYSTEM_ORIG   # restore

WITHOUT refusal sentence: The passage does not provide information about the hostel fee for international students. The relevant information is found in [1]: "Hostel accommodation is available for students whose home is outside the city, subject to availability." However, this statement does not specify any particular cost associated with hostel accommodation. Therefore, based solely on the given context, we cannot determine what the hostel fee is for international students.


## Task 5 — Hybrid Search: BM25 + Reciprocal Rank Fusion

In [14]:
def tokenize(text):
    return re.findall(r"\w[\w-]*", text.lower())

bm25 = BM25Okapi([tokenize(c) for c in chunks])

def bm25_retrieve(question, k=3):
    scores = bm25.get_scores(tokenize(question))
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]

def rrf(ranked_lists, k_const=60):
    """ranked_lists: list of lists of chunk indices, best first -> {chunk index: RRF score}"""
    fused = {}
    for ranked in ranked_lists:
        for rank, idx in enumerate(ranked, start=1):
            fused[idx] = fused.get(idx, 0.0) + 1.0 / (k_const + rank)
    return fused

def hybrid_retrieve(question, k=3, depth=10):
    dense = [i for i, _ in retrieve(question, depth)]
    sparse = [i for i, _ in bm25_retrieve(question, depth)]
    fused = rrf([dense, sparse])
    best = sorted(fused, key=fused.get, reverse=True)[:k]
    return [(i, fused[i]) for i in best]

q = "F-4015"
for name, fn in [("dense", retrieve), ("bm25", bm25_retrieve), ("hybrid", hybrid_retrieve)]:
    print(name.ljust(7), [i for i, _ in fn(q, 3)], " evidence in top-3:",
          any("F-4015" in chunks[i] for i, _ in fn(q, 3)))
print("\nHybrid Recall@3 =", recall_at_k(hybrid_retrieve, chunks, k=3))

dense   [4, 6, 3]  evidence in top-3: True
bm25    [4, 0, 2]  evidence in top-3: True
hybrid  [4, 3, 2]  evidence in top-3: True

Hybrid Recall@3 = 1.0


## Task 6 — HyDE: Search with a Hypothetical Answer

In [15]:
def hyde_retrieve(question, k=3):
    hypo = generate([{"role": "user", "content":
        "Write a short (2-3 sentence) passage, in the style of a university student handbook, "
        f"that answers this question.\n\nQuestion: {question}\n\nPassage:"}], max_new_tokens=120)
    h = embed([hypo])[0]                   # embed the hypothetical passage, not the question
    scores = chunk_emb @ h
    top = np.argsort(-scores)[:k]
    print("Hypothetical passage:", hypo, "\n")
    return [(int(i), float(scores[i])) for i in top]

def rank_of_evidence(results, evidence):
    for pos, (i, _) in enumerate(results, start=1):
        if evidence in chunks[i]:
            return pos
    return None

vague = "Can I get my money back if I drop out?"
print("plain search: evidence at rank", rank_of_evidence(retrieve(vague, len(chunks)), "refunded in full"))
print("HyDE search : evidence at rank", rank_of_evidence(hyde_retrieve(vague, len(chunks)), "refunded in full"))

plain search: evidence at rank 2
Hypothetical passage: Yes, you can typically get your money back if you decide to withdraw from a course or program before it ends. However, specific policies may vary depending on the institution and the type of financial aid you received. Always check with your school's financial aid office for details regarding refunds and withdrawals. 

HyDE search : evidence at rank 2


## Task 7 — Lost in the Middle
Give the model many passages, only one of which contains the answer, and move that passage from the first
position to the middle and to the last position.

In [16]:
small = [s for s in re.split(r"(?<=[.])\s+", HANDBOOK) if len(s.split()) > 4]   # one sentence per passage
print(len(small), "sentence passages")
TESTS = [("What is the library fine for an overdue book, per day?", "fifty rupees", "fifty"),
         ("Within how many days must medical leave be submitted?", "within seven days", "seven"),
         ("How many courses may a student on probation register for?", "maximum of four courses", "four")]

def lost_in_middle(question, evidence, keyword, position):
    gold = next(c for c in small if evidence in c)
    distractors = [c for c in small if evidence not in c]
    passages = distractors[:position] + [gold] + distractors[position:]
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": build_prompt(question, passages)}]
    answer = generate(messages, max_new_tokens=60)
    return keyword.lower() in answer.lower(), len(passages)

for q, ev, kw in TESTS:
    n = len(small)
    row = {pos_name: lost_in_middle(q, ev, kw, pos)[0]
           for pos_name, pos in [("first", 0), ("middle", (n - 1) // 2), ("last", n - 1)]}
    print(q[:55].ljust(56), row)

41 sentence passages
What is the library fine for an overdue book, per day?   {'first': True, 'middle': True, 'last': True}
Within how many days must medical leave be submitted?    {'first': True, 'middle': True, 'last': True}
How many courses may a student on probation register fo  {'first': True, 'middle': True, 'last': True}


## Task 8 (Challenge) — Agentic RAG with a Support Check
The model now **decides** when to search. It must reply with exactly one line, either
`SEARCH: <query>` or `ANSWER: <answer with citations>`. Your harness runs the searches and feeds the results back.

In [17]:
AGENT_SYSTEM = ("You answer questions about a university handbook. You cannot see the handbook directly. "
                "Reply with exactly ONE line, either\n"
                "SEARCH: <a short search query>\n"
                "or\n"
                "ANSWER: <final answer, citing passages as [1], [2]>\n"
                "Search first if you need information. Search again with a different query if the results do not help.")

def agent(question, max_steps=4):
    messages = [{"role": "system", "content": AGENT_SYSTEM}, {"role": "user", "content": question}]
    seen = []                                         # all passages shown to the model, in order
    for step in range(max_steps):
        reply = generate(messages, max_new_tokens=120).strip().splitlines()[0]
        print(f"step {step}: {reply}")
        messages.append({"role": "assistant", "content": reply})
        if reply.upper().startswith("ANSWER:"):
            return reply[7:].strip(), seen
        if reply.upper().startswith("SEARCH:"):
            query = reply[7:].strip()
            results = hybrid_retrieve(query, 3)
            new = [chunks[i] for i, _ in results]
            first_no = len(seen) + 1                      # numbering continues across searches
            seen.extend(new)
            shown = "\n".join(f"[{first_no + j}] {p}" for j, p in enumerate(new))
            messages.append({"role": "user", "content":
                f"Search results:\n{shown}\n\nNow reply with SEARCH: or ANSWER: (one line)."})
        else:
            messages.append({"role": "user", "content": "Please reply with SEARCH: or ANSWER: only."})
    return "Stopped: step limit reached.", seen

def check_support(answer, passages):
    """A Self-RAG-style check, done by the harness: is each sentence supported by the passages?"""
    context = "\n".join(passages)
    for sentence in re.split(r"(?<=[.!?])\s+", answer):
        if not sentence.strip():
            continue
        verdict = generate([{"role": "user", "content":
            f"Context:\n{context}\n\nSentence: {sentence}\n\n"
            "Is the sentence fully supported by the context? Answer only yes or no."}], max_new_tokens=3)
        print(("SUPPORTED   " if verdict.lower().startswith("yes") else "UNSUPPORTED ") + sentence)

ans, passages = agent("If I withdraw in the first week, how much of my fee do I get back, and which form do I use?")
print("\nFINAL:", ans, "\n")
check_support(ans, passages)

step 0: SEARCH: Withdrawal policy details
step 1: SEARCH: Withdrawal policy details
step 2: ANSWER: If you withdraw in the first week, you receive a 75% refund of your tuition fees. Use the "Withdrawal Form" to initiate the process.

FINAL: If you withdraw in the first week, you receive a 75% refund of your tuition fees. Use the "Withdrawal Form" to initiate the process. 

SUPPORTED   If you withdraw in the first week, you receive a 75% refund of your tuition fees.
SUPPORTED   Use the "Withdrawal Form" to initiate the process.


**Task 8(c): a question that needs no handbook**

In [18]:
ans2, p2 = agent("What is 2+2?")
print("FINAL:", ans2, "| passages seen:", len(p2))

step 0: SEARCH: What is 2+2?
step 1: SEARCH:
step 2: ANSWER: 2+2 equals 4.
FINAL: 2+2 equals 4. | passages seen: 6
